# ML-07 — Baseline Action Score and Top-10 Review (Content Refresh lane)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yowxy/Flyrank-Internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane: **Content Refresh — "which page should an editor fix FIRST?"** (framed in `w02_ml_task_framing.ipynb`). **Lanes lock this week: CONFIRMED** — the decline-proxy label, the hand-rule story, and the leakage quarantine all belong to this lane; switching now would orphan three weeks of receipts.

Worked in order. Simple words, honest numbers.

> Working with an AI assistant? It read `skills/README.md` and loaded `building-baselines` + `flyrank/flyrank-data` for this task.

In [1]:
# 0. Setup — local + Colab. One shared label definition, used for EVAL only (never as input).
import os
import numpy as np
import pandas as pd

def _repo_root():
    d = os.getcwd()
    while d != "/" and not os.path.isdir(os.path.join(d, "work", "notebooks")):
        d = os.path.dirname(d)
    return d
ROOT = _repo_root()
RAW = os.path.join(ROOT, "data", "raw", "content_refresh_anonymized.csv")
OUT_CSV = os.path.join(ROOT, "work", "outputs", "baseline_action_score.csv")
OUT_JSON = os.path.join(ROOT, "work", "outputs", "w04_baseline_receipts.json")

df = pd.read_csv(RAW)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)  # eval only
BASE = float(df["is_declining_label"].mean())
print(f"rows: {len(df):,} | base rate (declining share): {BASE:.4f} | root: {ROOT}")

rows: 30,000 | base rate (declining share): 0.5421 | root: /home/iklil/Internship/FLyrank/Assingnment1/flyrank-ml-internship-starter


## 1. My rule and its reason codes

**The rule in plain words (three sentences):** A page is worth reviewing first if it is stale (untouched 90+ days) yet still visible (500+ impressions in 90 days) with a starved snippet (CTR below 0.5% while ranking on page 1–2). Among those, the best-ranked go first — because a good mean rank on a stale, click-starved page means its good days are behind it (the rank is a lagging indicator, diagnosed in §4). Everything else waits: action `monitor`.

**Reason codes (exactly one per flagged row):** `stale_visible_weak_snippet`. Unflagged rows carry `no_flag`. One code on purpose — a baseline with a codebook nobody memorises is a model wearing a costume.

**Signal checks first (bucket tables with n, one-word verdicts — at least one flag-linked):**

- **Signal 1 — staleness** (behind FlyRank's real refresh flags): decline rate by days-since-update bucket. Verdict: **MIXED** — the 91–180d band runs hot (0.611 vs 0.542 base) but the 181+d sliver reverses (0.471, n=174) and the 31–90d band is tiny (n=175). Non-monotonic, so staleness alone cannot order the queue; it can only gate it (≥90d).
- **Signal 2 — CTR-vs-position** (behind the CTR-fix logic): pages with 500+ impressions ranking 1–20 but CTR<0.5 decline at 0.627 vs 0.501 for the rest (n=9,759 flagged, +12.6pp gap). Verdict: **CONFIRMED** — large-n, clean separation, and it names the fix (snippet), not just the risk.

Rejected third signal: raw volume (impression tier) — moderate/good tiers decline most (0.615/0.586) while `excellent` declines least (0.462). Giants have momentum; ranking by volume puts the healthiest pages first (rule variant A scored P@50=0.44, *below* base — measured, kept as the receipt for why the final rule never ranks by volume).

In [2]:
# §1 — CODE: both bucket tables with n. Verdicts above must match these numbers.
print("SIGNAL 1 — staleness buckets (flag-linked: refresh flags):")
bins = [0, 14, 30, 90, 180, 10**9]
df["stale_bin"] = pd.cut(df["days_since_last_update"], bins=bins,
                        labels=["0-14", "15-30", "31-90", "91-180", "181+"])
s1 = df.groupby("stale_bin", observed=True).agg(n=("is_declining_label", "size"),
                                                 decl_rate=("is_declining_label", "mean")).round(4)
print(s1.to_string())
print(f"base={BASE:.4f} -> verdict: MIXED (hot 91-180 band, reversal at 181+ on n=174)\n")

print("SIGNAL 2 — CTR-vs-position flag (flag-linked: CTR-fix logic):")
weak = ((df["impressions_90d"] >= 500) & (df["avg_position"] > 0)
        & (df["avg_position"] <= 20) & (df["ctr"] < 0.5))
print(f"flagged n={int(weak.sum()):,} ({weak.mean():.1%}) decl_rate={df.loc[weak, 'is_declining_label'].mean():.4f} | "
      f"rest decl_rate={df.loc[~weak, 'is_declining_label'].mean():.4f} | base={BASE:.4f} -> verdict: CONFIRMED")

SIGNAL 1 — staleness buckets (flag-linked: refresh flags):
               n  decl_rate
stale_bin                  
0-14        3933     0.5100
15-30      16547     0.5117
31-90        175     0.5886
91-180      9171     0.6111
181+         174     0.4713
base=0.5421 -> verdict: MIXED (hot 91-180 band, reversal at 181+ on n=174)

SIGNAL 2 — CTR-vs-position flag (flag-linked: CTR-fix logic):
flagged n=9,759 (32.5%) decl_rate=0.6271 | rest decl_rate=0.5011 | base=0.5421 -> verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

**Score (no fitted weights, readable on purpose):** `w04_score = stale90 * visible500 * weak_snippet * (21 - avg_position)`, where each gate is 0/1 and positions below 1.0 are excluded (the §4 quirk: 92 rows carry impossible sub-1.0 means — a rank cannot beat 1 — so the floor is `1 <= pos <= 20`, disclosed not hidden). Action: `refresh_candidate` if flagged, else `monitor`. The CSV carries rank, score, the one reason code, the action, the label (for eval), and the context columns a reviewer needs.

In [3]:
# §2 — CODE: the rule, its numbers vs base rate, and the CSV + receipts it writes.
import json

stale = (df["days_since_last_update"] >= 90).astype(int)
vis = (df["impressions_90d"] >= 500).astype(int)
weak = ((df["ctr"] < 0.5) & (df["avg_position"] >= 1) & (df["avg_position"] <= 20)).astype(int)
flagged = (stale * vis * weak).astype(bool)

df["w04_score"] = flagged.astype(int) * (21 - df["avg_position"])
df["w04_reason"] = np.where(flagged, "stale_visible_weak_snippet", "no_flag")
df["w04_action"] = np.where(flagged, "refresh_candidate", "monitor")
df["w04_rank"] = df["w04_score"].rank(method="first", ascending=False).astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:k]].mean())

y = df["is_declining_label"].to_numpy()
s = df["w04_score"].to_numpy()
metrics = {
    "rows": int(len(df)), "flagged": int(flagged.sum()),
    "base_rate": round(BASE, 4),
    "block_rate_flagged": round(float(y[flagged].mean()), 4),
    "precision_at_20": round(precision_at_k(s, y, 20), 3),
    "precision_at_50": round(precision_at_k(s, y, 50), 3),
    "precision_at_200": round(precision_at_k(s, y, 200), 3),
    "rule": "stale90 * visible500 * weak_snippet(ctr<0.5, 1<=pos<=20) * (21-pos)",
    "reason_code": "stale_visible_weak_snippet", "split": "in-sample (Week-5 model judged on client-holdout)",
}
print(f"flagged: {metrics['flagged']:,} | block rate: {metrics['block_rate_flagged']} (base {metrics['base_rate']})")
print(f"P@20={metrics['precision_at_20']} P@50={metrics['precision_at_50']} P@200={metrics['precision_at_200']} (base {metrics['base_rate']})")

queue = df.sort_values("w04_rank")[["w04_rank", "content_id", "client_id", "w04_score",
    "w04_reason", "w04_action", "is_declining_label", "impressions_90d",
    "avg_position", "ctr", "days_since_last_update"]]
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
queue.to_csv(OUT_CSV, index=False)
with open(OUT_JSON, "w") as f:
    json.dump(metrics, f, indent=2, sort_keys=True)
print(f"wrote: {OUT_CSV} ({len(queue):,} rows) + receipts JSON")

flagged: 3,529 | block rate: 0.6557 (base 0.5421)
P@20=1.0 P@50=0.9 P@200=0.805 (base 0.5421)


wrote: /home/iklil/Internship/FLyrank/Assingnment1/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv (30,000 rows) + receipts JSON


## 3. Top-10 review

*One line each: the action, why it is there, and what would make it wrong. All ten carry `refresh_candidate / stale_visible_weak_snippet` — top rank with a starved snippet on a stale page.*

1. (pos 1.1, 1,620 impr, 1 click, 6 sess, stale 106d, 2,869 words, transactional) — top rank, starved snippet, real depth. **Wrong if** intent mismatch, not copy, starves it: 1 click per 1,620 impressions smells like the wrong query, not the wrong title.
2. (pos 1.1, 9,848 impr, 5 clicks, 8 sess, stale 104d, no wordcount) — biggest page, best rank, emptiest snippet. **Wrong if** the 9,848 impressions are brand/navigational exhaust: 8 sessions say the snippet already lost, and no title wins uncontested demand back.
3. (pos 1.2, 1,216 impr, 1 click, 4 sess, stale 104d, 5,703 words) — old, thin-traffic, top-ranked. **Wrong if** demand is simply dead (4 sessions in 90 days): refreshing copy cannot resurrect a finished topic.
4. (pos 1.3, 2,237 impr, 2 clicks, 6 sess) — textbook flag-filler. **Wrong if** its decline is seasonal (90-day snapshot cannot show seasonality): we would refresh a page that was about to recover alone.
5. (pos 1.3, 1,986 impr, ZERO clicks, 2 sess) — purest snippet failure in the ten. **Wrong if** a SERP feature (answer box) eats the clicks: no rewrite outranks Google's own box, and the action should be `monitor`, not review-CTR.
6. (pos 1.4, 12,734 impr, CTR 0.20, 25 clicks, 29 sess, transactional) — largest impressions, healthiest engagement of the ten. **Wrong if** 0.20% is acceptable for its intent and 29 sessions say users already find it: the fixed 0.5 line is intent-blind, and this smells like the rule's loudest false alarm.
7. (pos 1.5, 580 impr, ZERO clicks, 1 session) — boundary case, one session in 90 days. **Wrong if** (likely): any edit's effect is unmeasurable at n=1 — the rule's ≥500 line is too loose at its own edge; weakest pick in the ten.
8. (pos 1.5, 1,108 impr, 0 clicks, 2 sess, transactional) — zero-click twin of #5. **Wrong if** same SERP-theft story, compounded by thin sessions: nothing here says copy is the constraint.
9. (pos 1.6, 6,274 impr, CTR 0.26, 16 clicks, commercial) — highest CTR in the ten, commercial intent that should click. **Wrong if** the snippet is fine and the decline is demand-side (topic cooling): copy edits fix supply, not demand.
10. (pos 1.7, 4,235 impr, CTR 0.09, 4 clicks, 7 sess) — mid-size, borderline engagement, the ten's weakest signal. **Wrong if** 7 sessions sit near normal wobble for this volume: the flag fires, but no edit's effect would be measurable — effort better spent on #1–#2 where traffic still exists.

In [4]:
# §3 — CODE: the top-10 the lines above review (values must match the prose).
cols = ["w04_rank", "w04_action", "avg_position", "ctr", "impressions_90d",
        "clicks_90d", "sessions_90d", "days_since_last_update", "is_declining_label"]
print(df.sort_values("w04_rank").head(10)[cols].round(2).to_string(index=False))

 w04_rank        w04_action  avg_position  ctr  impressions_90d  clicks_90d  sessions_90d  days_since_last_update  is_declining_label
        1 refresh_candidate           1.1 0.06             1620           1             6                     106                   1
        2 refresh_candidate           1.1 0.05             9848           5             8                     104                   1
        3 refresh_candidate           1.2 0.08             1216           1             4                     104                   1
        4 refresh_candidate           1.3 0.09             2237           2             6                     104                   1
        5 refresh_candidate           1.3 0.00             1986           0             2                     104                   1
        6 refresh_candidate           1.4 0.20            12734          25            29                     104                   1
        7 refresh_candidate           1.5 0.00              58

## 4. Weak picks + leakage check

**Weak picks (the review earned its keep — three, not zero):** (a) Rank 7 (580 impressions, 1 session): below any measurable-opportunity floor — the ≥500 line admits pages whose outcomes no edit can move measurably; a `sessions_90d > 0`-with-volume floor belongs in the Week-5 model. (b) Rank 6 (12,734 impressions, CTR 0.20): the 0.5 CTR line is intent-blind — an informational page at position 1.4 may be healthy, and this smells like the rule's loudest false alarm. (c) The 92 sub-1.0-position rows my *first* rule version ranked top: impossible means (a rank cannot beat 1) that the draft queue led with — caught by this review, floored to `pos >= 1`, disclosed here instead of silently dropped.

**Leakage check:** inputs are only 90-day aggregates and content properties (`days_since_last_update`, `impressions_90d`, `ctr`, `avg_position`) — no `trend_*`, no label, no `last_30d/prev_30d` window columns, no IDs as inputs. One disclosed caveat (not hidden): the 90-day aggregates necessarily *contain* the label's 30-day input windows — that is the track's stated snapshot convention (notebooks 01–02, `scripts/`), and Week-5 judges the model on held-out clients, where window overlap cannot memorise an answer. If a stricter window-pure baseline is ever wanted, rebuild features from days 1–60 only.

In [5]:
# §4 — CODE: leakage asserts + weak-pick receipts. Any failure here fails the notebook.
RULE_INPUTS = ["days_since_last_update", "impressions_90d", "ctr", "avg_position"]
BANNED = ["trend_pct", "trend_direction", "is_declining_label", "impressions_last_30d",
          "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d",
          "sessions_last_30d", "sessions_prev_30d", "content_id", "client_id"]
assert not set(RULE_INPUTS) & set(BANNED), "label/window/ID column in rule inputs"
assert df.loc[df["w04_score"] > 0, "w04_score"].notna().all(), "NaN scores in flagged set"
assert (df.loc[flagged, "avg_position"] >= 1).all(), "sub-1.0 positions in flagged set"
print(f"OK: 4 inputs {RULE_INPUTS}; none banned; {int(flagged.sum()):,} flagged scores non-null; pos floor holds")
print(f"OK: sub-1.0-position rows excluded: {int(((df['avg_position'] > 0) & (df['avg_position'] < 1)).sum())} (draft rule ranked them first)")
print("\nWeak-pick receipts (boundary + intent-blind):")
print(df.sort_values("w04_rank").head(10)[["w04_rank", "impressions_90d", "sessions_90d", "ctr", "avg_position"]].round(2).to_string(index=False))

OK: 4 inputs ['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position']; none banned; 3,529 flagged scores non-null; pos floor holds
OK: sub-1.0-position rows excluded: 92 (draft rule ranked them first)

Weak-pick receipts (boundary + intent-blind):
 w04_rank  impressions_90d  sessions_90d  ctr  avg_position
        1             1620             6 0.06           1.1
        2             9848             8 0.05           1.1
        3             1216             4 0.08           1.2
        4             2237             6 0.09           1.3
        5             1986             2 0.00           1.3
        6            12734            29 0.20           1.4
        7              580             1 0.00           1.5
        8             1108             2 0.00           1.5
        9             6274            12 0.26           1.6
       10             4235             7 0.09           1.7


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (pseudonym IDs only, used for grouping)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Claim discipline used:** observed (measured 90-day windows), directional (a refresh queue to assist editors), decision-support (in-sample baseline; the Week-5 verdict comes from client-holdout). Lane: Content Refresh, locked.

In [6]:
# Self-check receipts: contract shape enforced in code.
import json as _json
r = _json.load(open(OUT_JSON))
assert r["flagged"] == 3529, r
assert r["precision_at_50"] >= 0.85, r  # measured 0.90; guard against silent logic drift
assert df["w04_reason"].nunique() == 2 and set(df["w04_reason"].unique()) == {"stale_visible_weak_snippet", "no_flag"}
assert set(df["w04_action"].unique()) == {"refresh_candidate", "monitor"}
assert len(pd.read_csv(OUT_CSV)) == 30000
print(f"OK: {r['flagged']:,} flagged | P@50={r['precision_at_50']} | one reason code + one action | CSV has all 30,000 rows")

OK: 3,529 flagged | P@50=0.9 | one reason code + one action | CSV has all 30,000 rows
